<a href="https://colab.research.google.com/github/christophermagno/christophermagno/blob/main/Projects/My%20Bookshelf/goodreads_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📚 Goodreads Bookshelf Project

## ⚒️ Tools Used
[![Python](https://img.shields.io/badge/Python-3776AB?logo=python&logoColor=fff)](#)
[![Pandas](https://img.shields.io/badge/Pandas-150458?logo=pandas&logoColor=fff)](#)
[![Jupyter](https://img.shields.io/badge/Jupyter-ffffff?logo=Jupyter)](#)
[![ETL](https://custom-icon-badges.demolab.com/badge/ETL-9370DB?logo=etl-logo&logoColor=fff)](#)
[![EDA](https://custom-icon-badges.demolab.com/badge/EDA-9370DB?logo=etl-logo&logoColor=fff)](#)
[![Tableau](https://custom-icon-badges.demolab.com/badge/Tableau-0176D3?logo=tableau&logoColor=fff)](#)
* Google API
* Open Library

## 📖 Project Summary
This project transforms raw **Goodreads export data** into a structured analytics dataset to examine long-term reading behavior, preferences, and trends. The goal was to demonstrate the ability to **extract third-party exported data**, resolve data quality issues, enrich incomplete datasets using **external APIs**, and deliver **insight-driven visualizations** in Tableau.

The analysis focuses on **reading volume, pace, author and genre preferences, and rating patterns over time**, converting personal data into a scalable analytical workflow similar to those used in consumer analytics and product usage analysis.

- **Tools:** Python, pandas, Google Books API, Open Library API, Tableau
- **Dataset Size:** 238 rows × 24 columns (post-cleaning)
- **Output:** Interactive Tableau dashboard + analytical summary

To view the Tableau visualization, please click [here](https://public.tableau.com/app/profile/christopher.magno/viz/MyLibrary_17654594548470/MyBookshelf).

In [ ]:
# TODO: Use KMeans clustering to consolidate into clear, logical categories
# TODO: Have radial chart be ordered by date.

In [ ]:
import re
import logging
import importlib
import requests
from pathlib import Path

from tqdm import tqdm
import pandas as pd

from google.colab import drive

# Project directory
PROJECT_DIR = Path('/content/drive/MyDrive/Colab Notebooks/christophermagno/Projects/My Bookshelf')

# Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s', force=True)
log = logging.getLogger('Goodreads Project')

# Google Drive
drive.mount('/content/drive')
path_raw = PROJECT_DIR / 'goodreads_library_export.csv'

log.info(f'{path_raw.name} exists: {path_raw.exists()}')

2026-01-18 05:03:15,121 - INFO - goodreads_library_export.csv exists: True


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Read in the goodreads data

In [ ]:
df = pd.read_csv(path_raw)

## Checking out the data

In [ ]:
df.head()

,Book Id,Title,Author,Author l-f,Additional Authors,ISBN,ISBN13,My Rating,Average Rating,Publisher,...,Date Read,Date Added,Bookshelves,Bookshelves with positions,Exclusive Shelf,My Review,Spoiler,Private Notes,Read Count,Owned Copies
0,18400112,"The Devil Wears Scrubs (Dr. Jane McGill, #1)",Freida McFadden,"McFadden, Freida",NaN,"=""""","=""""",3,3.44,Hollywood Upstairs Publishing,...,NaN,2025/12/01,NaN,NaN,read,NaN,NaN,NaN,1,0
1,62047984,Yellowface,R.F. Kuang,"Kuang, R.F.",NaN,"=""""","=""""",4,3.73,William Morrow,...,NaN,2023/12/10,NaN,NaN,read,NaN,NaN,NaN,1,0
2,58416952,"The Will of the Many (Hierarchy, #1)",James Islington,"Islington, James",NaN,"=""1982141190""","=""9781982141196""",0,4.60,Gallery / Saga Press,...,NaN,2025/12/01,to-read,to-read (#52),to-read,NaN,NaN,NaN,0,0
3,20886354,"Skin Deep (Legion, #2)",Brandon Sanderson,"Sanderson, Brandon",Jon Foster,"=""""","=""""",4,4.12,Subterranean Press,...,2025/11/07,2025/11/07,NaN,NaN,read,NaN,NaN,NaN,1,0
4,58778536,Do Not Disturb,Freida McFadden,"McFadden, Freida",NaN,"=""""","=""""",3,3.90,Hollywood Upstairs Publishing,...,2025/11/13,2025/11/18,NaN,NaN,read,NaN,NaN,NaN,1,0


## Creating helper functions to clean ISBN code
They're currently stored as ="" or ="isbn#"

In [ ]:
def clean_isbn(isbn):
    return re.sub(r'["=]', '', isbn)

## There were no genres stored within the Goodreads dataset, so I'll be using GoogleAPI and Open Library to gather genre data for the books using the ISBN13 book code.

In [ ]:
def request_openlibrary_book_data(isbn):
    isbn = clean_isbn(isbn)
    url = f"https://openlibrary.org/isbn/{isbn}.json"
    response = requests.get(url)
    if not response:
        return {}

    response = response.json()

    # Get book "works" metadata (contains subjects)
    works_key = response.get("works", [{}])[0].get("key")
    if not works_key:
        return {}

    works_data = requests.get(f"https://openlibrary.org{works_key}.json").json()
    return works_data
request_openlibrary_book_data(df.loc[2, 'ISBN13'])

{'type': {'key': '/type/work'},
 'title': 'The Will of the Many',
 'authors': [{'author': {'key': '/authors/OL7468631A'},
   'type': {'key': '/type/author_role'}}],
 'key': '/works/OL31088394W',
 'subjects': ['series:Hierarchy', 'genre:high fantasy'],
 'description': "The Catenan Republic—the Hierarchy—may rule the world, but they do not know everything.\r\n\r\nI tell them my name is Vis Telimus. I tell them I was orphaned three years ago, and that only good fortune has got me into their most prestigious school. I tell them that, when I graduate, I will allow my strength and drive—what they call Will—to be leeched away and added to the power of those above me, as everyone must do.\r\n\r\nI tell them that I belong, and they believe me.\r\n\r\nBut the truth is that I have been sent to the Academy to solve a murder. To search for an ancient weapon. To uncover secrets that may tear the Republic apart.\r\n\r\nAnd that I will never cede my Will to the empire that executed my family.\r\n\r\nT

In [ ]:
def request_googleapi_book_data(isbn):
    isbn = clean_isbn(isbn)
    url = f"https://www.googleapis.com/books/v1/volumes?q=isbn:{isbn}"
    response = requests.get(url).json()
    if not 'items' in response:
        return {}
    return response['items'][0]
request_googleapi_book_data(df.loc[2, 'ISBN13'])

{'kind': 'books#volume',
 'id': 'jlGUEAAAQBAJ',
 'etag': 'QszM2Y0drG4',
 'selfLink': 'https://www.googleapis.com/books/v1/volumes/jlGUEAAAQBAJ',
 'volumeInfo': {'title': 'The Will of the Many',
  'authors': ['James Islington'],
  'publisher': 'Simon and Schuster',
  'publishedDate': '2023-05-23',
  'description': 'At the elite Catenan Academy, a young fugitive uncovers layered mysteries and world-changing secrets in this “brilliant and gut-churning masterpiece” (Library Journal, starred review) by the internationally bestselling author of The Licanius Trilogy, James Islington. The Catenan Republic—the Hierarchy—may rule the world now, but they do not know everything. I tell them my name is Vis Telimus. I tell them I was orphaned after a tragic accident three years ago, and that good fortune alone has led to my acceptance into their most prestigious school. I tell them that once I graduate, I will gladly join the rest of civilized society in allowing my strength, my drive, and my focus—

### Data that's useful in the gathered data from GoodleAPI and Open Library.
- volumeInfo
    - authors
    - title
    - publisher
    - publishDate
    - description
    - pageCount
    - categories [list]
    - maturityRating
    - imagelinks (do i want images?): [dict]
    - saleInfo
        - country
        - retailPrice
- subjects

In [ ]:
def clean_categories_list(categories, ignore=('collectionID', 'nyt')):

    new_categories = set()
    if not categories:
        return list(new_categories)

    for i in range(len(categories)):
        # Split and find categories to add to new_categories set
        for category in categories[i].split(', '):
            found = False
            for item in ignore:
                if item in category:
                    found = True
                    break
            if found:
                continue
            new_categories.add(category.title().replace('_', ' '))

    return list(new_categories)


### Final function to organize data for categories (genres)

In [ ]:
def get_book_data(isbn):
    """
    Function to gather additional data from google API and Open Library

    :param isbn: Book ISBN code
    :type isbn: str
    :return: dict
    """

    # Get and clean the ISBN
    isbn = clean_isbn(isbn)

    result = {'Categories': []}

    # Google API query
    googleapi_data = request_googleapi_book_data(isbn)
    if googleapi_data:
        result.update(
            {'Published Date': googleapi_data['volumeInfo'].get('publishedDate'),
            'Categories': clean_categories_list(googleapi_data['volumeInfo'].get('categories')),
            'Maturity Rating': googleapi_data['volumeInfo'].get('maturityRating'),
            'Description': googleapi_data['volumeInfo'].get('description'),
            'Country': googleapi_data['saleInfo'].get('country'),
            'Retail Price': googleapi_data['saleInfo'].get('retailPrice', {}).get('amount'),
            'Currency': googleapi_data['saleInfo'].get('retailPrice', {}).get('currencyCode')}
        )

    # Open AI query
    openai_categories_data = request_openlibrary_book_data(isbn).get('subjects', [])
    if openai_categories_data:
        result['Categories'].extend(clean_categories_list(openai_categories_data))

    # Sort the categories
    result['Categories'] = sorted(set((result['Categories'])))

    return result


### Sample request to see what data I get

In [ ]:
data = get_book_data(df.loc[2, 'ISBN13'])
data

{'Categories': ['Fiction', 'Genre:High Fantasy', 'Series:Hierarchy'],
 'Published Date': '2023-05-23',
 'Maturity Rating': 'NOT_MATURE',
 'Description': 'At the elite Catenan Academy, a young fugitive uncovers layered mysteries and world-changing secrets in this “brilliant and gut-churning masterpiece” (Library Journal, starred review) by the internationally bestselling author of The Licanius Trilogy, James Islington. The Catenan Republic—the Hierarchy—may rule the world now, but they do not know everything. I tell them my name is Vis Telimus. I tell them I was orphaned after a tragic accident three years ago, and that good fortune alone has led to my acceptance into their most prestigious school. I tell them that once I graduate, I will gladly join the rest of civilized society in allowing my strength, my drive, and my focus—what they call Will—to be leeched away and added to the power of those above me, as millions already do. As all must eventually do. I tell them that I belong, and

## Add the extra data into the DataFrame

In [ ]:
df.columns

Index(['Book Id', 'Title', 'Author', 'Author l-f', 'Additional Authors',
       'ISBN', 'ISBN13', 'My Rating', 'Average Rating', 'Publisher', 'Binding',
       'Number of Pages', 'Year Published', 'Original Publication Year',
       'Date Read', 'Date Added', 'Bookshelves', 'Bookshelves with positions',
       'Exclusive Shelf', 'My Review', 'Spoiler', 'Private Notes',
       'Read Count', 'Owned Copies'],
      dtype='object')

In [ ]:
categories = []

# New df to concatenate
concat_df = pd.DataFrame(df['Book Id'])

# Categories dataframe
for idx, row in tqdm(df.iterrows(), total=len(df)):
    isbn = clean_isbn(row['ISBN13'])
    book_data = get_book_data(isbn)
    if book_data:
        for key, value in book_data.items():
            if key == 'Categories':
                for category in book_data['Categories']:
                    categories.append(
                        {'Book Id': row['Book Id'], 'Category': category}
                    )
                continue
            concat_df.at[idx, key] = value

concat_df

100%|██████████| 237/237 [03:39<00:00,  1.08it/s]


,Book Id,Published Date,Maturity Rating,Description,Country,Retail Price,Currency
0,18400112,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
1,62047984,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
2,58416952,2023-05-23,NOT_MATURE,"At the elite Catenan Academy, a young fugitive...",US,16.99,USD
3,20886354,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
4,58778536,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
...,...,...,...,...,...,...,...
232,40389527,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
233,38389488,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
234,39863488,2019-02-05,NOT_MATURE,"INSTANT NEW YORK TIMES BESTSELLER ""I love Jane...",US,None,None
235,43848929,2019-09-10,NOT_MATURE,"Malcolm Gladwell, host of the podcast Revision...",US,None,None


## Merge and create copy of new DataFrame

In [ ]:
df_cp1 = df.merge(concat_df, how='left', on='Book Id')

In [ ]:
df_cp1.columns

Index(['Book Id', 'Title', 'Author', 'Author l-f', 'Additional Authors',
       'ISBN', 'ISBN13', 'My Rating', 'Average Rating', 'Publisher', 'Binding',
       'Number of Pages', 'Year Published', 'Original Publication Year',
       'Date Read', 'Date Added', 'Bookshelves', 'Bookshelves with positions',
       'Exclusive Shelf', 'My Review', 'Spoiler', 'Private Notes',
       'Read Count', 'Owned Copies', 'Published Date', 'Maturity Rating',
       'Description', 'Country', 'Retail Price', 'Currency'],
      dtype='object')

In [ ]:
df_cp1.head()

,Book Id,Title,Author,Author l-f,Additional Authors,ISBN,ISBN13,My Rating,Average Rating,Publisher,...,Spoiler,Private Notes,Read Count,Owned Copies,Published Date,Maturity Rating,Description,Country,Retail Price,Currency
0,18400112,"The Devil Wears Scrubs (Dr. Jane McGill, #1)",Freida McFadden,"McFadden, Freida",NaN,"=""""","=""""",3,3.44,Hollywood Upstairs Publishing,...,NaN,NaN,1,0,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
1,62047984,Yellowface,R.F. Kuang,"Kuang, R.F.",NaN,"=""""","=""""",4,3.73,William Morrow,...,NaN,NaN,1,0,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
2,58416952,"The Will of the Many (Hierarchy, #1)",James Islington,"Islington, James",NaN,"=""1982141190""","=""9781982141196""",0,4.60,Gallery / Saga Press,...,NaN,NaN,0,0,2023-05-23,NOT_MATURE,"At the elite Catenan Academy, a young fugitive...",US,16.99,USD
3,20886354,"Skin Deep (Legion, #2)",Brandon Sanderson,"Sanderson, Brandon",Jon Foster,"=""""","=""""",4,4.12,Subterranean Press,...,NaN,NaN,1,0,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
4,58778536,Do Not Disturb,Freida McFadden,"McFadden, Freida",NaN,"=""""","=""""",3,3.90,Hollywood Upstairs Publishing,...,NaN,NaN,1,0,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None


## Create a new Dataframe for the categories

In [ ]:
df_categories = pd.DataFrame(categories)
df_categories

,Book Id,Category
0,18400112,Language Arts & Disciplines
1,62047984,Language Arts & Disciplines
2,58416952,Fiction
3,58416952,Genre:High Fantasy
4,58416952,Series:Hierarchy
...,...,...
3808,43848929,Social Science
3809,43848929,Strangers
3810,43848929,Threat (Psychology)
3811,43848929,Trust


### Check genre counts

In [ ]:
category_counts = df_categories['Category'].value_counts()
category_counts

,count
Category,
Fiction,129
General,79
New York Times Bestseller,77
Language Arts & Disciplines,74
Fantasy,48
...,...
Fathers And Sons,1
Fathers And Sons Fiction; Fiction Dystopian; Fiction Science Fiction General; American Fiction (Fict,1
Fathers And Sons--Fiction,1


In [ ]:
len(category_counts)

2052

### 2052 categories is too much... let's prune the list
Remove some specific generic genres and any genre that only has 1

In [ ]:
categories_to_drop = [
    'Fiction',
    'General',
    'New York Times Bestseller',
    'Language Arts & Disciplines',
    'Romans',
    'Large Type Books',
    'New York Times Reviewed',
    'Reading Level-Grade 11',
    'Reading Level-Grade 12',
    'Open Library Staff Picks',
    'Long Now Manual For Civilization',
    'Ficción',
    'Novela',
    'Reading Level-Grade 10',
    'Reading Level-Grade 9'
] + category_counts[category_counts <= 3].index.tolist()
sorted(categories_to_drop)

['& Magic',
 '1000Blackgirlbooks',
 '153.4/4',
 '18.05 English Literature',
 '18.06 Anglo-American Literature',
 '1847-1912',
 '1866-1946',
 '1905-1997',
 '1911-1993',
 '1919-2010',
 '1939-1945',
 '1954-',
 '1961-1975',
 '1991',
 '2000S',
 '2001-02',
 '2005',
 '20Th Century',
 '21St Century Film And Literature',
 '306.81',
 '362.82/092 B',
 '813.5 S 3-8',
 '813/.3',
 '813/.54',
 '813/.6',
 '823.912',
 '823/.912',
 '883.01',
 '883/.001',
 '883/.01',
 'A Wrinkle In Time',
 'Ability',
 'Abridged Audio - Misc. Nonfiction',
 'Abuelos (Hombres)',
 'Abuse',
 'Academic Literacy',
 'Accessible Book',
 'Accounting',
 'Acculturation',
 'Achab (Personnage Fictif)',
 'Achille (Mythologie Grecque)',
 'Achilles',
 'Achilles (Greek Mythology)',
 'Achilles (Greek Mythology) In Literature',
 'Achilles (Greek Mythology)--Fiction',
 'Achilles (Greek Mythology)--Poetry',
 'Achilles--Poetry',
 'Action & Adventure - General',
 'Action And Adventure',
 'Actors',
 'Actors And Actresses',
 'Addiction',
 'Adoles

In [ ]:
drop_mask = df_categories[df_categories['Category'].isin(categories_to_drop)]
df_categories = df_categories.drop(drop_mask.index, axis=0)
df_categories

,Book Id,Category
8,22878967,Epic
9,22878967,Fantasy
10,22878967,Fantasy Fiction
13,22878967,Imaginary Wars And Battles
15,37640636,Action & Adventure
...,...,...
3796,39863488,Psychological
3798,39863488,Thrillers
3800,43848929,Conduct Of Life
3802,43848929,Interpersonal Relations


## Cleaning main book Dataframe

In [ ]:
df_cp1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 237 entries, 0 to 236
Data columns (total 30 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Book Id                     237 non-null    int64  
 1   Title                       237 non-null    object 
 2   Author                      237 non-null    object 
 3   Author l-f                  237 non-null    object 
 4   Additional Authors          55 non-null     object 
 5   ISBN                        237 non-null    object 
 6   ISBN13                      237 non-null    object 
 7   My Rating                   237 non-null    int64  
 8   Average Rating              237 non-null    float64
 9   Publisher                   233 non-null    object 
 10  Binding                     237 non-null    object 
 11  Number of Pages             236 non-null    float64
 12  Year Published              236 non-null    float64
 13  Original Publication Year   235 non

### Clean the `ISBN` columns

In [ ]:
for col, series in df_cp1[['ISBN', 'ISBN13']].items():
    df_cp1[col] = df_cp1[col].apply(clean_isbn)

### Fixing some author's names in `Author` column

In [ ]:
to_replace = {
    'Abraham   Verghese': 'Abraham Verghese',
    'Stephen        King': 'Stephen King'
}

for author, fixed in to_replace.items():
    df['Author'] = df['Author'].str.replace(author, fixed)

### Filling null values for `Additional Authors`

In [ ]:
df_cp1['Additional Authors'] = df_cp1['Additional Authors'].fillna('None')
df_cp1['Additional Authors'].isnull().sum()

np.int64(0)

### Filling null values for `Number of Pages`

In [ ]:
df_cp1['Number of Pages'] = df_cp1['Number of Pages'].fillna(0)
df_cp1['Additional Authors'].isnull().sum()
df_cp1['Number of Pages'] = df_cp1['Number of Pages'].astype(int)
df_cp1['Number of Pages'].isnull().sum()

np.int64(0)

### Was replaced with Published Date from GoogleAPI (had yyyy/mm/dd instead of just the year)

In [ ]:
df_cp1 = df_cp1.drop('Year Published', axis=1)

In [ ]:
df_cp1['Original Publication Year'] = df_cp1['Original Publication Year'].fillna(0)
df_cp1['Original Publication Year'] = df_cp1['Original Publication Year'].astype(int)

In [ ]:
df_cp1['Date Read'] = pd.to_datetime(df_cp1['Date Read'])
df_cp1['Date Added'] = pd.to_datetime(df_cp1['Date Added'])
df_cp1['Published Date'] = pd.to_datetime(df_cp1['Published Date'], format='mixed')

In [ ]:
df_cp1['Retail Price'] = df_cp1['Retail Price'].astype(float)

In [ ]:
df_cp1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 237 entries, 0 to 236
Data columns (total 29 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Book Id                     237 non-null    int64         
 1   Title                       237 non-null    object        
 2   Author                      237 non-null    object        
 3   Author l-f                  237 non-null    object        
 4   Additional Authors          237 non-null    object        
 5   ISBN                        237 non-null    object        
 6   ISBN13                      237 non-null    object        
 7   My Rating                   237 non-null    int64         
 8   Average Rating              237 non-null    float64       
 9   Publisher                   233 non-null    object        
 10  Binding                     237 non-null    object        
 11  Number of Pages             237 non-null    int64         

## Export Data

In [ ]:
df_cp1.to_csv(PROJECT_DIR / 'my_books_data.csv')
df_categories.to_csv(PROJECT_DIR / 'my_books_categories.csv')